# 02 Model evaluation

Baselines vs gradient boosting on the time-ordered holdout (last 20% of simulated sequences). Set `SG_ML_RUN` to a run id (default: the newest run in `data/runs/`).

In [ ]:
import os

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sg_ml.data import ROOT

FIG_DIR = ROOT / "packages" / "ml" / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})


def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=300)
    plt.show()
    plt.close(fig)

In [ ]:
import json

runs_dir = ROOT / "data" / "runs"
run_id = (
    os.environ.get("SG_ML_RUN")
    or sorted(p.name for p in runs_dir.iterdir() if (p / "metrics.json").exists())[-1]
)
run = json.loads((runs_dir / run_id / "metrics.json").read_text(encoding="utf-8"))
H = pd.read_parquet(runs_dir / run_id / "holdout.parquet")
print(run_id, run["feature_set"], "tuned:", run["tuned"], "holdout days:", len(H))
pd.DataFrame(run["metrics"]["regression"]).T

## Baselines vs models (MAE)

In [ ]:
reg = pd.DataFrame(run["metrics"]["regression"]).T.sort_values("mae")
fig, ax = plt.subplots(figsize=(8, 3.5))
colors = ["#c8553d" if n.startswith("baseline") else "#2b7a78" for n in reg.index]
ax.barh(reg.index, reg["mae"], color=colors)
ax.invert_yaxis()
ax.set_xlabel("MAE (overtime hours)")
imp = run["metrics"]["improvement_vs_best_baseline"]
ax.set_title(
    f"p50 is {1 - imp['ratio']:.0%} better than the best baseline ({imp['best_baseline']})"
)
save(fig, "eval_mae_comparison")

## Residuals

In [ ]:
res = H["actual"] - H["p50"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(H["p50"], res, s=6)
axes[0].axhline(0, color="k", lw=0.8)
axes[0].set_xlabel("Predicted (p50)")
axes[0].set_ylabel("Actual - predicted")
axes[1].hist(res, bins=50)
axes[1].set_xlabel("Residual (hours)")
save(fig, "eval_residuals")

## 80% interval coverage

In [ ]:
S = H.sort_values("p50").reset_index(drop=True)
inside = (S["actual"] >= S["p10"]) & (S["actual"] <= S["p90"])
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].fill_between(S.index, S["p10"], S["p90"], alpha=0.3, label="p10 to p90")
axes[0].plot(S.index, S["p50"], lw=1, label="p50")
axes[0].scatter(
    S.index[~inside], S["actual"][~inside], s=6, color="#c8553d", label="actual outside"
)
axes[0].set_xlabel("Holdout days sorted by p50")
axes[0].legend()
bins = pd.qcut(S["p50"].rank(method="first"), 5, labels=False)
cov = inside.groupby(bins).mean()
axes[1].bar(cov.index.astype(str), cov.values)
axes[1].axhline(0.8, color="k", ls="--")
axes[1].set_ylim(0, 1)
axes[1].set_xlabel("p50 quintile")
axes[1].set_title(f"Coverage {inside.mean():.1%} (target 80%)")
save(fig, "eval_interval_coverage")

## Burden classes: confusion matrices

In [ ]:
clf = run["metrics"]["classification"]
fig, axes = plt.subplots(1, len(clf), figsize=(5 * len(clf), 4))
for ax, (name, m) in zip(np.atleast_1d(axes), clf.items(), strict=True):
    cm = np.array(m["confusion"])
    ax.imshow(cm, cmap="Blues")
    for i in range(3):
        for j in range(3):
            ax.text(j, i, cm[i, j], ha="center", va="center")
    ax.set_xticks(range(3), m["labels"])
    ax.set_yticks(range(3), m["labels"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"{name}\nmacro F1 {m['macro_f1']:.2f}, Brier {m['brier']:.3f}", fontsize=10)
save(fig, "eval_confusion")

## Full vs core feature set

The core set uses only Tier 1 inputs, the calendar, lags and patients per morning staff: what a department can provide every day without the Tier 2 log.

In [ ]:
fsets = pd.DataFrame(run["metrics"]["feature_sets"]).T[
    ["mae", "rmse", "coverage_80", "mean_width_80"]
]
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(fsets.index, fsets["mae"], color=["#2b7a78", "#8a9ba8"])
ax.set_ylabel("MAE (hours)")
save(fig, "eval_full_vs_core")
fsets